In [ ]:
from pathlib import Path
import openslide

he_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")

print("Formato detectado:", openslide.OpenSlide.detect_format(str(he_path)))

slide = openslide.OpenSlide(str(he_path))
print("Vendor:", slide.properties.get("openslide.vendor"))
print("Niveles:", slide.level_count)
print("Dimensiones nivel 0:", slide.dimensions)
print("Associated images:", list(slide.associated_images.keys()))

In [ ]:
import matplotlib.pyplot as plt

level = slide.level_count - 1
img = slide.read_region((0, 0), level, slide.level_dimensions[level]).convert("RGB")

plt.figure(figsize=(8, 12))
plt.imshow(img)
plt.axis("off")
plt.title("H&E preview")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

for name in slide.associated_images.keys():
    img = slide.associated_images[name]
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(name)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

# coordenadas en nivel 0
x, y = 40000, 120000
w, h = 3000, 3000
level = 0

region = slide.read_region((x, y), level, (w, h)).convert("RGB")

plt.figure(figsize=(8, 8))
plt.imshow(region)
plt.axis("off")
plt.title(f"Region level {level}")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

# máscara simple: fondo casi negro
mask = thumb_np.mean(axis=2) > 20

ys, xs = np.where(mask)
xmin, xmax = xs.min(), xs.max()
ymin, ymax = ys.min(), ys.max()

plt.figure(figsize=(6, 10))
plt.imshow(thumb_np)
plt.plot([xmin, xmax, xmax, xmin, xmin],
         [ymin, ymin, ymax, ymax, ymin], 'r-')
plt.axis("off")
plt.title("Bounding box en thumbnail")
plt.show()

# escalar a nivel 0
thumb_w, thumb_h = thumb.size
full_w, full_h = slide.dimensions

scale_x = full_w / thumb_w
scale_y = full_h / thumb_h

XMIN = int(xmin * scale_x)
XMAX = int(xmax * scale_x)
YMIN = int(ymin * scale_y)
YMAX = int(ymax * scale_y)

print(XMIN, XMAX, YMIN, YMAX)

In [ ]:
print(slide.level_dimensions)
print(slide.level_downsamples)

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# --- rutas ---
slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
xml_path = Path(r"Datos\SB013\SB013\H&E_python\H&E.xml")

# --- abrir slide ---
slide = openslide.OpenSlide(str(slide_path))

# --- leer XML ---
tree = ET.parse(xml_path)
root = tree.getroot()

source = root.find("source")
destination = root.find("destination")
annotations = destination.find("annotations")

# tamaño fuente/destino descrito en el XML
src_roi = source.find("region_of_interest")
dst_roi = destination.find("region_of_interest")

src_w = int(src_roi.attrib["width"])
src_h = int(src_roi.attrib["height"])
dst_w = int(dst_roi.attrib["width"])
dst_h = int(dst_roi.attrib["height"])

print("XML source ROI:", src_w, src_h)
print("XML destination ROI:", dst_w, dst_h)

# --- sacar thumbnail o nivel bajo ---
thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

thumb_w, thumb_h = thumb.size

# escalado desde coordenadas XML (nivel 0) a thumbnail
scale_x = thumb_w / src_w
scale_y = thumb_h / src_h

# colores por etiqueta
label_colors = {
    "Tumour tissue": "yellow",
    "Healthy tissue": "lime",
    "Fibrosis": "cyan",
    "Ductal ctasia": "orange",
    "Greeen": "magenta",
    "RED": "red",
}

fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(thumb_np)

for ann in annotations.findall("annotation"):
    name = ann.attrib.get("name", "annotation")
    pts = []

    for p in ann.findall("p"):
        x = float(p.attrib["x"]) * scale_x
        y = float(p.attrib["y"]) * scale_y
        pts.append((x, y))

    if len(pts) >= 3:
        poly = Polygon(
            pts,
            closed=True,
            fill=False,
            edgecolor=label_colors.get(name, "white"),
            linewidth=2,
            label=name
        )
        ax.add_patch(poly)

        # texto en el primer punto
        ax.text(
            pts[0][0], pts[0][1],
            name,
            color=label_colors.get(name, "white"),
            fontsize=8,
            bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
        )

ax.set_title("Thumbnail + anotaciones")
ax.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
xml_path = Path(r"Datos\SB013\SB013\H&E_python\H&E.xml")

slide = openslide.OpenSlide(str(slide_path))

tree = ET.parse(xml_path)
root = tree.getroot()

annotations = root.find("destination").find("annotations")

# región a leer en coordenadas nivel 0
x0, y0 = 35000, 100000
w0, h0 = 50000, 70000
level = 3

# tamaño de la región en ese nivel
downsample = slide.level_downsamples[level]
region_size = (int(w0 / downsample), int(h0 / downsample))

region = slide.read_region((x0, y0), level, region_size).convert("RGB")
region_np = np.array(region)

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(region_np)

label_colors = {
    "Tumour tissue": "yellow",
    "Healthy tissue": "lime",
    "Fibrosis": "cyan",
    "Ductal ctasia": "orange",
    "Greeen": "magenta",
    "RED": "red",
}

for ann in annotations.findall("annotation"):
    name = ann.attrib.get("name", "annotation")
    pts = []

    for p in ann.findall("p"):
        x = float(p.attrib["x"])
        y = float(p.attrib["y"])

        # quedarnos solo con puntos cercanos a la región
        xr = (x - x0) / downsample
        yr = (y - y0) / downsample
        pts.append((xr, yr))

    # dibujar solo si el polígono cae en la ventana
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]

    if max(xs) < 0 or max(ys) < 0 or min(xs) > region_size[0] or min(ys) > region_size[1]:
        continue

    if len(pts) >= 3:
        poly = Polygon(
            pts,
            closed=True,
            fill=False,
            edgecolor=label_colors.get(name, "white"),
            linewidth=2
        )
        ax.add_patch(poly)

        ax.text(
            pts[0][0], pts[0][1],
            name,
            color=label_colors.get(name, "white"),
            fontsize=9,
            bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
        )

ax.set_title(f"Region level {level} + anotaciones")
ax.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# --- ruta XML ---
xml_path = Path(r"Datos\SB013\SB013\H&E_python\H&E.xml")

# --- leer XML ---
tree = ET.parse(xml_path)
root = tree.getroot()

annotations = root.find("destination").find("annotations")

rows = []

for i, ann in enumerate(annotations.findall("annotation"), start=1):
    name = ann.attrib.get("name", "annotation")
    ann_type = ann.attrib.get("type", "")
    color_bgr = ann.attrib.get("color_bgr", "")

    pts = []
    for p in ann.findall("p"):
        x = float(p.attrib["x"])
        y = float(p.attrib["y"])
        pts.append((x, y))

    if not pts:
        continue

    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]

    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)

    width = xmax - xmin
    height = ymax - ymin
    cx = (xmin + xmax) / 2
    cy = (ymin + ymax) / 2

    rows.append({
        "roi_id": i,
        "label": name,
        "type": ann_type,
        "color_bgr": color_bgr,
        "n_points": len(pts),
        "xmin": xmin,
        "ymin": ymin,
        "xmax": xmax,
        "ymax": ymax,
        "width": width,
        "height": height,
        "center_x": cx,
        "center_y": cy,
        "points": pts,   # lista completa de vértices
    })

df_rois = pd.DataFrame(rows)

print(df_rois[[
    "roi_id", "label", "type", "n_points",
    "xmin", "ymin", "xmax", "ymax",
    "width", "height", "center_x", "center_y"
]])

In [ ]:
point_rows = []

for _, row in df_rois.iterrows():
    for j, (x, y) in enumerate(row["points"], start=1):
        point_rows.append({
            "roi_id": row["roi_id"],
            "label": row["label"],
            "point_id": j,
            "x": x,
            "y": y
        })

df_points = pd.DataFrame(point_rows)

print(df_points)

In [ ]:
for _, row in df_rois.iterrows():
    print(f"\nROI {row['roi_id']} - {row['label']}")
    print(f"  bbox: xmin={row['xmin']}, ymin={row['ymin']}, xmax={row['xmax']}, ymax={row['ymax']}")
    print(f"  center: ({row['center_x']:.1f}, {row['center_y']:.1f})")
    print(f"  n_points: {row['n_points']}")
    print("  points:")
    for j, (x, y) in enumerate(row["points"], start=1):
        print(f"    {j}: ({x}, {y})")

Segmentacion

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

from PIL import Image
import cv2

# --- ruta slide ---
slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")

slide = openslide.OpenSlide(str(slide_path))

# --- cargar thumbnail ---
thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

# ---------------------------------------------------
# 1) máscara tejido vs background
# ---------------------------------------------------
# fondo negro -> tejido = píxeles no oscuros
gray = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2GRAY)

# umbral simple; puedes ajustar 15, 20, 25...
_, mask = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)

# limpiar ruido y cerrar huecos pequeños
kernel = np.ones((5, 5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

# ---------------------------------------------------
# 2) quedarnos con el componente conectado más grande
# ---------------------------------------------------
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)

if num_labels <= 1:
    raise ValueError("No se ha detectado tejido. Ajusta el umbral.")

# ignorar label 0 = fondo
largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
specimen_mask = np.uint8(labels == largest_label) * 255

# ---------------------------------------------------
# 3) contorno externo del espécimen
# ---------------------------------------------------
contours, _ = cv2.findContours(specimen_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if not contours:
    raise ValueError("No se encontró contorno del espécimen.")

contour = max(contours, key=cv2.contourArea)  # contorno mayor

# simplificar contorno para tener menos puntos
epsilon = 0.002 * cv2.arcLength(contour, True)
contour_approx = cv2.approxPolyDP(contour, epsilon, True)

# puntos en coordenadas thumbnail
thumb_points = [(int(pt[0][0]), int(pt[0][1])) for pt in contour_approx]

# ---------------------------------------------------
# 4) escalar a coordenadas nivel 0
# ---------------------------------------------------
thumb_w, thumb_h = thumb.size
full_w, full_h = slide.dimensions

scale_x = full_w / thumb_w
scale_y = full_h / thumb_h

roi_level0 = [(int(x * scale_x), int(y * scale_y)) for x, y in thumb_points]

print("Número de puntos del ROI:", len(roi_level0))
print("Primeros puntos:", roi_level0[:10])

# ---------------------------------------------------
# 5) visualizar sobre thumbnail
# ---------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(thumb_np)

poly = Polygon(thumb_points, closed=True, fill=False, edgecolor="yellow", linewidth=2)
ax.add_patch(poly)

ax.set_title("ROI del espécimen sobre thumbnail")
ax.axis("off")
plt.show()

no funciona

In [ ]:
print(slide.level_dimensions)
print(slide.level_downsamples)

In [ ]:
from pathlib import Path
import openslide
import matplotlib.pyplot as plt

slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

# región en coordenadas de nivel 0
x, y = 60000, 130000
w, h = 3000, 3000

region = slide.read_region((x, y), 0, (w, h)).convert("RGB")

plt.figure(figsize=(10, 10))
plt.imshow(region)
plt.axis("off")
plt.title(f"Nivel 0 | x={x}, y={y}, w={w}, h={h}")
plt.show()

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

slide_path = Path(r"Datos\SB013\SB013\H&E_python\SB013.mrxs")
slide = openslide.OpenSlide(str(slide_path))

# ROI en level 0
x, y = 60000, 130000
w, h = 3000, 3000

# recorte real
region = slide.read_region((x, y), 0, (w, h)).convert("RGB")

# thumbnail
thumb = slide.associated_images["thumbnail"].convert("RGB")
thumb_np = np.array(thumb)

thumb_w, thumb_h = thumb.size
full_w, full_h = slide.dimensions

scale_x = thumb_w / full_w
scale_y = thumb_h / full_h

x_t = x * scale_x
y_t = y * scale_y
w_t = w * scale_x
h_t = h * scale_y

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# izquierda: vista global
axes[0].imshow(thumb_np)
rect = Rectangle((x_t, y_t), w_t, h_t, fill=False, edgecolor="yellow", linewidth=2)
axes[0].add_patch(rect)
axes[0].set_title("Ubicación del ROI en la imagen global")
axes[0].axis("off")

# derecha: recorte
axes[1].imshow(region)
axes[1].set_title(f"Recorte extraído\nlevel=0, x={x}, y={y}, w={w}, h={h}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# region = PIL image de tu read_region(...)
img = np.array(region)                 # RGB
img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

H, S, V = cv2.split(hsv)

# -----------------------------
# 1) máscara de zonas blancas
# -----------------------------
# blanco = poca saturación + mucho brillo
mask_white = ((S < 40) & (V > 200)).astype(np.uint8) * 255

# limpiar ruido
kernel = np.ones((3, 3), np.uint8)
mask_white = cv2.morphologyEx(mask_white, cv2.MORPH_OPEN, kernel)
mask_white = cv2.morphologyEx(mask_white, cv2.MORPH_CLOSE, kernel)

# rellenar pequeños agujeros
mask_white = cv2.medianBlur(mask_white, 5)

# -----------------------------
# 2) máscara de tejido rosado
# -----------------------------
# lo más simple: tejido = no blanco
mask_tissue = cv2.bitwise_not(mask_white)

# opcional: quedarte con zonas con cierta saturación
mask_pink = ((S >= 40) & (V < 245)).astype(np.uint8) * 255
mask_pink = cv2.bitwise_and(mask_pink, mask_tissue)

# visualizar
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
axes[0].imshow(img)
axes[0].set_title("Imagen original")
axes[0].axis("off")

axes[1].imshow(mask_white, cmap="gray")
axes[1].set_title("Contornos / regiones blancas")
axes[1].axis("off")

axes[2].imshow(mask_pink, cmap="gray")
axes[2].set_title("Tejido rosado")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
L, A, B = cv2.split(lab)

# blanco: L alta, cromaticidad moderada-baja
mask_white_lab = ((L > 210) & (A < 145) & (B < 150)).astype(np.uint8) * 255

kernel = np.ones((3, 3), np.uint8)
mask_white_lab = cv2.morphologyEx(mask_white_lab, cv2.MORPH_OPEN, kernel)
mask_white_lab = cv2.morphologyEx(mask_white_lab, cv2.MORPH_CLOSE, kernel)

plt.figure(figsize=(6, 6))
plt.imshow(mask_white_lab, cmap="gray")
plt.axis("off")
plt.title("Blancos con Lab")
plt.show()